# Tree Search Comparison



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import numpy as np
import statsmodels.api as sm

# Configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)

# MSAs
## Expected TKF root length

In [ ]:
import numpy as np

def simulate_tkf_root_length(n_experiments=1000, lam=0.03, mu=0.0303, r=0.8):
    """
    Simulates the root length of a TKF model.
    The number of links follows a geometric distribution (success = stopping).
    The length of each fragment (if length > 0) follows a geometric distribution.
    """
    # 1. Probability of "stopping" a link (success in Geometric)
    prob_stop_links = 1.0 - (lam / mu)
    
    # 2. Probability of "stopping" a fragment extension
    # In TKF92/TKF, fragment length ~ Geometric(1-r)
    prob_stop_fragment = 1.0 - r
    
    results = []
    
    for _ in range(n_experiments):
        # Equivalent to sample_num_root_links
        # np.random.geometric(p) returns value in [1, inf). Subtract 1 to match 0-indexed links
        num_links = np.random.geometric(prob_stop_links) - 1
        
        total_length = 0
        for _ in range(num_links):
            # Equivalent to sample_fragment_length
            # Fragments in TKF are at least 1 unit long
            frag_len = np.random.geometric(prob_stop_fragment)
            total_length += frag_len
            
        results.append(total_length)
    
    # Analytical expected value: (lambda/mu) / ((1 - lambda/mu) * (1 - r))
    expected_val = (lam / mu) / ((1 - (lam / mu)) * (1 - r))
    simulated_val = np.mean(results)
    
    return {
        "Simulated Average": simulated_val,
        "Analytical Expected": expected_val,
        "Difference": abs(simulated_val - expected_val),
        "Raw Data": results
    }

# Run experiment
sim_results = simulate_tkf_root_length(n_experiments=100000)
{k: v for k, v in sim_results.items() if k != "Raw Data"}

## Loading MSA simulation data

In [ ]:
# Load MSA summary data
msa_df = pd.read_csv('results/msa_summary.tsv', sep='\t')

## MSA simulation plots stratified

In [ ]:
# Configuration for stratification
tool_hue = 'msa_sim_tool'

# Define a consistent palette colors for the tools
unique_tools = msa_df[tool_hue].unique()
palette = dict(zip(unique_tools, sns.color_palette("husl", len(unique_tools))))

# Ensure numeric types
msa_metrics = ['msa_len', 'gap%', 'gap_col%', 'avg_gap_len', 'species']
for col in msa_metrics:
    if col in msa_df.columns:
        msa_df[col] = pd.to_numeric(msa_df[col], errors='coerce')

# Create a 2x2 grid for the 4 metrics stratified by Tree Size (Species)
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
axes_flat = axes.flatten()

# Metrics to plot
plot_items = [
    ('msa_len', 'Alignment Lengths'),
    ('gap%', 'Gap Percentages'),
    ('gap_col%', 'Percentage of Gap Columns'),
    ('avg_gap_len', 'Average Gap Length')
]

# Generate the 2x2 grid
for i, (metric, label) in enumerate(plot_items):
    sns.boxplot(
        data=msa_df, 
        x='species', 
        y=metric, 
        hue=tool_hue, 
        palette=palette, 
        ax=axes_flat[i]
    )
    axes_flat[i].set_title(f'{label} by Tree Size (Species)')
    axes_flat[i].set_xlabel('Number of Species')
    
    # Only show legend on the first plot to save space, or use a global one
    if i != 1:
        axes_flat[i].get_legend().remove()

plt.tight_layout()
plt.show()

# Inference

In [ ]:
# Load data
df = pd.read_csv('results/summary.tsv', sep='\t')

# Note: The 'gap_strategy' column seems to contain the model name (PIP/TKF92)
model_col = 'gap_strategy'

# Ensure numeric types for metrics
metrics = ['robinson_foulds', 'kuhner_felsenstein', 'start_robinson_foulds', 'start_kuhner_felsenstein', 'runtime_seconds', 'log_likelihood', 'alignment_length', 'gap_percentage', 'species']
for col in metrics:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} results.")
df

In [ ]:
# mean of the iqtree rf
iqtree_mean_rf = df[(df["inference_tool"] == "jati") & (df["gap_strategy"] == "TKF92")]["rf"].mean()
print(f"Mean RF for IQ-TREE: {iqtree_mean_rf}")

## 1. Topological and Branch Length Accuracy

The **Robinson-Foulds (RF)** distance measures the symmetric difference between the inferred tree and the true tree topology. The **Kuhner-Felsenstein (KF)** distance accounts for both topology and branch length estimation accuracy. **Lower is better for both.**

In [ ]:
# Update accuracy logic to include specific search strategies for both RF and KF
def map_accuracy_strategy(row):
    gs = str(row['gap_strategy']).lower()
    ms = str(row['move_strategy']).lower() if pd.notnull(row['move_strategy']) else ""
    it = str(row['inference_tool']).lower() if 'inference_tool' in row and pd.notnull(row['inference_tool']) else ""
    
    # 1. IQ-TREE (checking inference_tool explicitly)
    if it == 'iqtree':
        return "IQ-TREE"
    
    # 2. TKF92 (always with NNI via jati)
    if 'tkf' in gs:
        return "TKF92"
    
    # 3. PIP (strictly split by NNI vs SPR via jati)
    if 'pip' in gs:
        if ms == "nni":
            return "PIP + NNI"
        if ms == "spr":
            return "PIP + SPR"       
    return None

# Filter out 'true_tree' rows before mapping and analysis
accuracy_filtered = df[df['inference_tool'] != 'true_tree'].copy()

# Apply mapping and filter out rows that don't match our criteria
accuracy_filtered['accuracy_strategy'] = accuracy_filtered.apply(map_accuracy_strategy, axis=1)
accuracy_df = accuracy_filtered[accuracy_filtered['accuracy_strategy'].notnull()].copy()

print(f"Removed {len(df) - len(accuracy_df)} rows (including true_tree and non-matching strategies).")

# Prepare data for RF and KF
metrics_to_plot = [('rf', 'start_rf', 'Robinson-Foulds Distance'), 
                   ('kf', 'start_kf', 'Kuhner-Felsenstein Distance')]

fig, axes = plt.subplots(1, 2, figsize=(10, 6))

for i, (main_col, start_col, title) in enumerate(metrics_to_plot):
    plot_df = pd.melt(accuracy_df, id_vars=['accuracy_strategy', 'species'], 
                      value_vars=[main_col, start_col], 
                      var_name='tree_type', value_name='dist_val')
    plot_df['model'] = plot_df.apply(lambda x: 'NJ (Start)' if x['tree_type'] == start_col else x['accuracy_strategy'], axis=1)
    
    sns.violinplot(data=plot_df, x='model', y='dist_val', hue='model', 
                   palette='Set2', inner="quart", legend=False, ax=axes[i])
    axes[i].set_title(title)
    axes[i].set_ylabel('Distance')
    axes[i].set_xlabel('Search Strategy')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Stratified Accuracy Analysis (RF and KF) by Tree Size
# Using the same mapping and filtering logic as defined in Cell 12

# Prepare data for RF and KF with species stratification
metrics_to_plot = [('rf', 'start_rf', 'Robinson-Foulds Distance'), 
                   ('kf', 'start_kf', 'Kuhner-Felsenstein Distance')]

# Create a figure for each metric, with facets for species
for i, (main_col, start_col, title) in enumerate(metrics_to_plot):
    # Melt data to compare inferred models with NJ Start tree
    plot_df = pd.melt(accuracy_df, id_vars=['accuracy_strategy', 'species'], 
                      value_vars=[main_col, start_col], 
                      var_name='tree_type', value_name='dist_val')
    
    # Create the 'model' category (NJ Start vs the specific search strategies)
    plot_df['model'] = plot_df.apply(lambda x: 'NJ (Start)' if x['tree_type'] == start_col else x['accuracy_strategy'], axis=1)
    
    # Plotting using FacetGrid to stratify by species
    # We use a height that allows the labels to be readable
    g = sns.FacetGrid(plot_df, col="species", height=4, aspect=0.8, sharey=False)
    g.map_dataframe(sns.violinplot, x='model', y='dist_val', hue='model', 
                    palette='Set2', inner="quart", legend=False)
    
    g.set_axis_labels("Search Strategy", "Distance")
    g.set_titles("Species: {col_name}")
    g.set_xticklabels(rotation=45)
    
    plt.subplots_adjust(top=0.85)
    g.fig.suptitle(f'{title} Stratified by Tree Size (Species)')
    plt.show()

In [ ]:
# Comparison of Accuracy by Simulation Tool (TKF vs. AliSim)
# Stratified by Species and Search Strategy

# msa_sim_tool is already present in accuracy_df. 
# We filter out 'true_tree' and unidentified strategies as done in cell 12/13.
plot_accuracy_df = accuracy_df.copy()

# Prepare metrics to plot
metrics_to_plot = [('rf', 'start_rf', 'Robinson-Foulds Distance'), 
                   ('kf', 'start_kf', 'Kuhner-Felsenstein Distance')]

for i, (main_col, start_col, title) in enumerate(metrics_to_plot):
    # Melt data to compare inferred models with NJ Start tree
    # We include msa_sim_tool to distinguish the simulation process
    plot_df = pd.melt(plot_accuracy_df, id_vars=['accuracy_strategy', 'species', 'msa_sim_tool'], 
                      value_vars=[main_col, start_col], 
                      var_name='tree_type', value_name='dist_val')
    
    # Create the 'model' category
    plot_df['model'] = plot_df.apply(lambda x: 'NJ (Start)' if x['tree_type'] == start_col else x['accuracy_strategy'], axis=1)
    
    # Create a FacetGrid: columns are Species
    g = sns.FacetGrid(plot_df, col="species", height=5, aspect=1.2, sharey=False)
    
    # Use 'hue' for the simulation tool to get side-by-side violins (split=True)
    g.map_dataframe(sns.violinplot, x='model', y='dist_val', hue='msa_sim_tool', 
                    palette='PRGn', inner="quart", split=True)
    
    g.set_axis_labels("Search Strategy", "Distance")
    g.set_titles("Species: {col_name}")
    g.set_xticklabels(rotation=45)
    g.add_legend(title="Simulation Tool")
    
    plt.subplots_adjust(top=0.85)
    g.fig.suptitle(f'{title}: TKF vs AliSim Simulations')
    plt.show()

## 3. Computational Complexity and Runtime Analysis

Comparing runtimes across model types and tree sizes.

In [ ]:
# Create a refined strategy column based on your specified categories
def map_search_strategy(row):
    gs = str(row['gap_strategy']).lower()
    ms = str(row['move_strategy']).lower() if pd.notnull(row['move_strategy']) else ""
    it = str(row['inference_tool']).lower() if 'inference_tool' in row and pd.notnull(row['inference_tool']) else ""
    
    # 1. IQ-TREE
    if it == 'iqtree':
        return "IQ-TREE"
    
    # 2. TKF92
    if 'tkf' in gs:
        return "TKF92"
    
    # 3. PIP (split by NNI vs SPR)
    if 'pip' in gs:
        if ms == "nni":
            return "PIP + NNI"
        if ms == "spr":
            return "PIP + SPR"
    
    return None

# Apply and filter
df['combined_strategy'] = df.apply(map_search_strategy, axis=1)
efficiency_filtered = df[df['combined_strategy'].notnull()].copy()

# Stratified Runtime Plot
g = sns.FacetGrid(efficiency_filtered, col="species", hue="combined_strategy", height=4, aspect=1, sharey=False)
g.map(sns.scatterplot, "msa_len", "runtime_seconds", s=80, alpha=0.6)
g.map(sns.regplot, "msa_len", "runtime_seconds", scatter=False)
g.add_legend(title="Search Strategy")
g.set_axis_labels("Alignment Length (bp)", "Runtime (seconds)")
plt.subplots_adjust(top=0.8)
g.fig.suptitle('Runtime vs Alignment Length: IQ-TREE vs TKF92 vs PIP (NNI/SPR)')
plt.show()

In [ ]:
# Filter out 'PIP + SPR' to compare only the faster strategies
efficiency_no_spr = efficiency_filtered[efficiency_filtered['combined_strategy'] != 'PIP + SPR'].copy()

# Stratified Runtime Plot (Excluding PIP + SPR)
g = sns.FacetGrid(efficiency_no_spr, col="species", hue="combined_strategy", height=4, aspect=1, sharey=False)
g.map(sns.scatterplot, "msa_len", "runtime_seconds", s=80, alpha=0.6)
g.map(sns.regplot, "msa_len", "runtime_seconds", scatter=False)
g.add_legend(title="Search Strategy")
g.set_axis_labels("Alignment Length (bp)", "Runtime (seconds)")
plt.subplots_adjust(top=0.8)
g.fig.suptitle('Runtime Benchmarking: IQ-TREE vs TKF92 vs PIP (NNI only)')
plt.show()

In [ ]:
# Diagnostic: Investigate TKF92 Bimodal Runtime at 64 Species
# We isolate TKF92 at 64 species and split it into two clusters based on the 100s threshold

tkf64 = efficiency_filtered[
    (efficiency_filtered['species'] == 64) & 
    (efficiency_filtered['combined_strategy'] == 'TKF92')
].copy()

# Define clusters based on observation
tkf64['runtime_cluster'] = tkf64['runtime_seconds'].apply(lambda x: 'Fast (<100s)' if x < 100 else 'Slow (>100s)')

# Dynamically identify the gap percentage column
gap_col = 'gap%' if 'gap%' in tkf64.columns else ('gap_percent' if 'gap_percent' in tkf64.columns else 'gap_percentage')

# Compare clusters across simulation tools and MSA properties
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Runtime vs Alignment Length
# Color by msa_sim_tool and Size by KF distance
sns.scatterplot(
    data=tkf64, 
    x='msa_len', 
    y='runtime_seconds', 
    hue='msa_sim_tool', 
    size='kf', 
    sizes=(20, 300), 
    alpha=0.7,
    palette='Set2', 
    ax=axes[0]
)
axes[0].set_title('Runtime vs Length (Size = KF Distance)')
axes[0].legend(title='Sim Tool / KF Dist', bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 2: Gap percentage comparison
if gap_col in tkf64.columns:
    sns.boxplot(
        data=tkf64, 
        x='runtime_cluster', 
        y=gap_col, 
        hue='runtime_cluster', 
        palette='Set2', 
        legend=False, 
        ax=axes[1]
    )
    axes[1].set_title(f'{gap_col} Comparison')
else:
    axes[1].text(0.5, 0.5, 'Gap Column Not Found', ha='center')

plt.tight_layout()
plt.show()

# Display summary statistics including KF distance
cols_to_summarize = ['msa_len', 'runtime_seconds', 'kf']
if gap_col in tkf64.columns:
    cols_to_summarize.append(gap_col)

tkf64.groupby('runtime_cluster')[cols_to_summarize].mean()

In [ ]:
# Histogram of Robinson-Foulds (RF) Distance for TKF92 at 64 Species
# Comparing the 'Fast' and 'Slow' runtime clusters with increased bin resolution

plt.figure(figsize=(10, 6))

# Plotting the histogram with KDE for each cluster and increased bins
sns.histplot(
    data=tkf64, 
    x='rf', 
    hue='runtime_cluster', 
    element='step', 
    kde=True, 
    palette='Set2',
    alpha=0.4,
    bins=30  # Increased bin number for higher resolution
)

plt.title('RF Distance Distribution for TKF92 (64 Species)')
plt.xlabel('Robinson-Foulds Distance')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Precise mean RF distance for each cluster
tkf64.groupby('runtime_cluster')['rf'].mean()